In [1]:
import requests
import pandas as pd
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

def fetch_worldbank_indicator(country_code, indicator_code, short_name):
    url = f"https://api.worldbank.org/v2/country/{country_code}/indicator/{indicator_code}"
    params = {
        "format": "json",
        "per_page": 20000
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    data = response.json()[1]
    
    rows = []
    for item in data:
        rows.append({
            "year": int(item["date"]),
            short_name: item["value"]
        })
    
    return pd.DataFrame(rows)

In [2]:
worldbank_indicators = {
    "gdp_per_capita": "NY.GDP.PCAP.CD",
    "poverty_headcount": "SI.POV.NAHC",
    "health_expenditure_gdp": "SH.XPD.CHEX.GD.ZS",
    "out_of_pocket_health_exp": "SH.XPD.OOPC.CH.ZS",
    "secondary_school_enrolment": "SE.SEC.ENRR",
    "urban_population_pct": "SP.URB.TOTL.IN.ZS",
    "internet_users_pct": "IT.NET.USER.ZS",
    "population_65plus_pct": "SP.POP.65UP.TO.ZS"
}

In [3]:
dfs = []

for short_name, code in worldbank_indicators.items():
    df = fetch_worldbank_indicator("IND", code, short_name)
    dfs.append(df)

determinants = dfs[0]

for df in dfs[1:]:
    determinants = determinants.merge(df, on="year", how="outer")

determinants = determinants.sort_values("year").reset_index(drop=True)

determinants.head()

,year,gdp_per_capita,poverty_headcount,health_expenditure_gdp,out_of_pocket_health_exp,secondary_school_enrolment,urban_population_pct,internet_users_pct,population_65plus_pct
0,1960,84.932808,NaN,NaN,NaN,NaN,17.925007,NaN,3.299052
1,1961,87.853861,NaN,NaN,NaN,NaN,18.024009,NaN,3.338526
2,1962,92.199958,NaN,NaN,NaN,NaN,18.143571,NaN,3.395098
3,1963,103.435021,NaN,NaN,NaN,NaN,18.283856,NaN,3.458202
4,1964,117.856431,NaN,NaN,NaN,NaN,18.443041,NaN,3.512548


In [4]:
determinants.to_csv(
    "../data/processed/india_determinants_panel.csv",
    index=False
)